# LeetCode 690 — Employee Importance bằng Python

Link đề: https://leetcode.com/problems/employee-importance/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Array, Hash Table

Skeleton LeetCode:

```python
"""
# Definition for Employee.
class Employee:
    def __init__(self, id: int, importance: int, subordinates: List[int]):
        self.id = id
        self.importance = importance
        self.subordinates = subordinates
"""

class Solution:
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        ...
```

Ghi chú: LeetCode tự cung cấp lớp `Employee` (đoạn chú thích ở trên). Trong notebook ta **tự dựng** lớp `Employee` với ba thuộc tính `id`, `importance`, `subordinates`, cùng hàm `make_employees` để chuyển từ dạng list `[[1,5,[2,3]], ...]` của đề sang đối tượng. Ví dụ 3, 4, 5 trong mục 2 do notebook tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách nhân viên, mỗi người có `id` duy nhất, `importance` (độ quan trọng, có thể **âm**) và danh sách `id` của các **cấp dưới trực tiếp**; với một `id` cho trước, tính **tổng độ quan trọng của người đó và mọi cấp dưới trực tiếp lẫn gián tiếp**.
2. Nắm các công cụ Python: định nghĩa `class` với `__init__` và thuộc tính, chuyển list đối tượng thành `dict` tra theo `id`, đệ quy, `collections.deque` cho BFS, ngăn xếp bằng `list`, `sum` với biểu thức sinh (generator), và giới hạn đệ quy `sys.setrecursionlimit`.
3. Đi từ **quét cả danh sách để tìm từng nhân viên (O(n²))** → **bảng băm + DFS đệ quy** → **bảng băm + BFS** → **DFS lặp dùng ngăn xếp (không sợ đệ quy sâu)** → **tính sẵn tổng cho mọi nhân viên từ lá lên gốc (Kahn)**.
4. Hiểu mấu chốt: dữ liệu thực chất là một **rừng cây** (mỗi người tối đa một sếp trực tiếp); tổng của một người = `importance` của chính họ + tổng của từng cấp dưới trực tiếp. Muốn truy cập nhanh theo `id` phải lập `dict {id: nhân viên}` — `id` **không** phải chỉ số trong list.
5. Tránh các lỗi kinh điển: dùng `employees[id]` như chỉ số, quên cộng chính người được hỏi, chỉ cộng cấp dưới **trực tiếp** (bỏ sót gián tiếp), sửa/xoá dữ liệu đầu vào khi duyệt, cộng nhầm `id` thay vì `importance`, và đệ quy quá sâu trên chuỗi chỉ huy dài.

## 2. Đề bài & ví dụ

Bạn có một cấu trúc dữ liệu thông tin nhân viên gồm: `id` duy nhất, giá trị `importance` (độ quan trọng) và danh sách `id` của các **cấp dưới trực tiếp** của mỗi nhân viên. Cụ thể, mảng `employees` có:

- `employees[i].id`: `id` của nhân viên thứ `i`;
- `employees[i].importance`: độ quan trọng của nhân viên thứ `i`;
- `employees[i].subordinates`: danh sách `id` cấp dưới trực tiếp của nhân viên thứ `i`.

Cho một số nguyên `id` là `id` của một nhân viên, trả về **tổng độ quan trọng** của nhân viên đó **và tất cả cấp dưới trực tiếp lẫn gián tiếp** của họ.

**Ví dụ 1**

- Input: `employees = [[1,5,[2,3]],[2,3,[]],[3,3,[]]]`, `id = 1`
- Output: `11`
- Giải thích: nhân viên 1 có độ quan trọng 5 và hai cấp dưới trực tiếp là 2 và 3, mỗi người 3. Tổng `5 + 3 + 3 = 11`.

**Ví dụ 2**

- Input: `employees = [[1,2,[5]],[5,-3,[]]]`, `id = 5`
- Output: `-3`
- Giải thích: nhân viên 5 có độ quan trọng `-3` và không có cấp dưới, nên tổng là `-3`.

**Ví dụ 3 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `employees = [[1,5,[2]],[2,3,[3]],[3,4,[]]]`, `id = 1`
- Output: `12` (cấp dưới **gián tiếp** `3` cũng được tính: `5 + 3 + 4`).

**Ví dụ 4 (tự đặt)**

- Input: `employees = [[1,5,[2]],[2,3,[3]],[3,4,[]]]`, `id = 2`
- Output: `7` (nhân viên giữa chuỗi: chỉ tính từ 2 trở xuống, **không** tính sếp 1).

**Ví dụ 5 (tự đặt)**

- Input: `employees = [[7,10,[]]]`, `id = 7`
- Output: `10` (một nhân viên duy nhất).

**Ràng buộc:**

- `1 <= employees.length <= 2000`
- `1 <= employees[i].id <= 2000`
- Mọi `employees[i].id` **đôi một khác nhau**.
- `-100 <= employees[i].importance <= 100`
- Một nhân viên có **tối đa một sếp trực tiếp** và có thể có nhiều cấp dưới.
- Các `id` trong `employees[i].subordinates` đều **hợp lệ** (có trong `employees`).

Lưu ý: `id` có thể thưa (ví dụ 1, 5, 2000) nên **không** dùng `id` làm chỉ số của list `employees`.

In [1]:
import sys
from collections import deque
import random
import time
from typing import List


class Employee:
    """Mô phỏng lớp Employee mà LeetCode cung cấp."""
    def __init__(self, id: int, importance: int, subordinates: List[int]):
        self.id = id
        self.importance = importance
        self.subordinates = subordinates

    def __repr__(self):
        return f"Employee({self.id}, {self.importance}, {self.subordinates})"


def make_employees(rows):
    """Chuyển list kiểu [[id, importance, [subordinates]], ...] thành list đối tượng Employee."""
    return [Employee(i, imp, list(subs)) for i, imp, subs in rows]


examples = [
    ([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]], 1, 11),
    ([[1, 2, [5]], [5, -3, []]], 5, -3),
    ([[1, 5, [2]], [2, 3, [3]], [3, 4, []]], 1, 12),
    ([[1, 5, [2]], [2, 3, [3]], [3, 4, []]], 2, 7),
    ([[7, 10, []]], 7, 10),
]
for rows, id_, expected in examples:
    print(make_employees(rows), "id =", id_, "-> kỳ vọng", expected)

[Employee(1, 5, [2, 3]), Employee(2, 3, []), Employee(3, 3, [])] id = 1 -> kỳ vọng 11
[Employee(1, 2, [5]), Employee(5, -3, [])] id = 5 -> kỳ vọng -3
[Employee(1, 5, [2]), Employee(2, 3, [3]), Employee(3, 4, [])] id = 1 -> kỳ vọng 12
[Employee(1, 5, [2]), Employee(2, 3, [3]), Employee(3, 4, [])] id = 2 -> kỳ vọng 7
[Employee(7, 10, [])] id = 7 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Lớp (class), đối tượng và thuộc tính

Một **lớp** là bản thiết kế cho một kiểu dữ liệu; `__init__` là hàm khởi tạo, `self.id`, `self.importance`, `self.subordinates` là **thuộc tính** của từng đối tượng. Truy cập bằng dấu chấm: `e.importance`. Lớp `Employee` ở trên mô phỏng đúng lớp của LeetCode.

In [2]:
e = Employee(1, 5, [2, 3])
print(e.id, e.importance, e.subordinates)
print(e)
e2 = Employee(2, 3, [])
print("e2 có cấp dưới không?", bool(e2.subordinates), "| số cấp dưới của e:", len(e.subordinates))

1 5 [2, 3]
Employee(1, 5, [2, 3])
e2 có cấp dưới không? False | số cấp dưới của e: 2


### 3.2. Từ list đối tượng sang `dict` tra theo `id`

Danh sách `employees` có thứ tự bất kỳ và `id` không liên quan tới vị trí. Để tra "nhân viên có `id = 5`" nhanh, ta lập `dict` bằng **dict comprehension**: `by_id = {e.id: e for e in employees}`. Sau đó `by_id[5]` là đối tượng cần tìm, mỗi lần tra trung bình `O(1)` thay vì quét cả list `O(n)`.

In [3]:
employees = make_employees([[1, 2, [5]], [5, -3, []]])
by_id = {e.id: e for e in employees}
print(by_id)
print(by_id[5].importance, "| employees[5] sẽ lỗi vì list chỉ có", len(employees), "phần tử")
try:
    employees[5]
except IndexError as err:
    print("lỗi:", type(err).__name__, "-", err)

{1: Employee(1, 2, [5]), 5: Employee(5, -3, [])}
-3 | employees[5] sẽ lỗi vì list chỉ có 2 phần tử
lỗi: IndexError - list index out of range


### 3.3. Cây, rừng và "cấp dưới trực tiếp / gián tiếp"

Mỗi người có **tối đa một sếp** nên quan hệ sếp–cấp dưới tạo thành một **cây** (hoặc nhiều cây tách rời: một **rừng**). Một người **không có sếp** là **gốc**; người **không có cấp dưới** là **lá**. Cấp dưới **gián tiếp** là cấp dưới của cấp dưới... Tập "người đó và mọi cấp dưới trực tiếp/gián tiếp" chính là **cây con (subtree)** gốc tại người đó.

Ví dụ 1 vẽ thành:

```
    1 (5)
   /     \
 2 (3)   3 (3)
```

In [4]:
def show_tree(employees, root_id, indent=0):
    by_id = {e.id: e for e in employees}
    e = by_id[root_id]
    print("  " * indent + f"{e.id} (importance={e.importance})")
    for s in e.subordinates:
        show_tree(employees, s, indent + 1)

show_tree(make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]), 1)

1 (importance=5)
  2 (importance=3)
    4 (importance=-2)
  3 (importance=3)


### 3.4. Đệ quy và giới hạn đệ quy

Đệ quy: hàm tự gọi chính nó cho bài toán nhỏ hơn. Ở đây: `tổng(người) = importance + Σ tổng(từng cấp dưới)`. Điều kiện dừng tự nhiên là người **không có cấp dưới** (tổng chỉ gồm chính họ).

Python giới hạn độ sâu đệ quy (mặc định khoảng **1000**). Một chuỗi chỉ huy dài 2000 người (mỗi người là sếp của người kế tiếp) cần độ sâu 2000 — vượt giới hạn mặc định! Ta sẽ gặp lại điều này ở mục 12–13.

In [5]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

print(depth(500))
try:
    depth(5000)
except RecursionError as err:
    print("lỗi:", type(err).__name__, "-", err)

giới hạn đệ quy mặc định: 1000
500
lỗi: RecursionError - maximum recursion depth exceeded


### 3.5. `deque` và BFS; ngăn xếp và DFS lặp

- **BFS** (duyệt theo tầng) dùng hàng đợi: lấy người ở **đầu**, thêm các cấp dưới vào **cuối**. `collections.deque` cho `append` / `popleft` đều `O(1)`.
- **DFS lặp** dùng **ngăn xếp** (`list` với `append` / `pop` ở cuối): lấy người ở **cuối**, thêm các cấp dưới vào cuối. Không dùng đệ quy nên không bị giới hạn độ sâu.

Cả hai cho **cùng tập người** (chỉ khác thứ tự thăm); vì ta chỉ cộng tổng nên thứ tự không quan trọng.

In [6]:
q = deque([1])
order = []
tree = {1: [2, 3], 2: [4], 3: [], 4: []}
while q:
    v = q.popleft()
    order.append(v)
    q.extend(tree[v])
print("BFS:", order)

stack = [1]
order = []
while stack:
    v = stack.pop()
    order.append(v)
    stack.extend(tree[v])
print("DFS lặp:", order)

BFS: [1, 2, 3, 4]
DFS lặp: [1, 3, 2, 4]


### 3.6. `sum` với biểu thức sinh (generator expression)

`sum(f(x) for x in danh_sach)` cộng các giá trị mà không cần tạo list trung gian; nếu danh sách rỗng thì `sum` trả `0` — rất tiện cho trường hợp "không có cấp dưới".

In [7]:
print(sum(x * x for x in [1, 2, 3]), sum(x for x in []), sum([-3]))
by_id = {e.id: e for e in make_employees([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]])}
print(by_id[1].importance + sum(by_id[s].importance for s in by_id[1].subordinates))

14 0 -3
11


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `employees = [[1,5,[2,3]],[2,3,[4]],[3,3,[]],[4,-2,[]]]`, hỏi `id = 1`.

Vẽ cây:

```
1 (5)
├─ 2 (3)
│   └─ 4 (-2)
└─ 3 (3)
```

Tính từ lá lên (hoặc từ trên xuống rồi cộng):

| người | importance | các cấp dưới trực tiếp | tổng cây con |
|---|---|---|---|
| 4 | -2 | — | -2 |
| 3 | 3 | — | 3 |
| 2 | 3 | 4 | 3 + (-2) = 1 |
| 1 | 5 | 2, 3 | 5 + 1 + 3 = 9 |

Đáp án `9`. Cách con người làm: **cầm danh sách, lần theo từng người của nhóm "sếp 1", đánh dấu những người đã tính** và cộng `importance` — không cần quan tâm họ nằm ở vị trí nào trong danh sách; chỉ cần có cách **tra nhanh một người theo `id`** (như tra danh bạ).

In [8]:
rows = [[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]
employees = make_employees(rows)
by_id = {e.id: e for e in employees}

def hand_total(id_):
    e = by_id[id_]
    parts = [by_id[s] for s in e.subordinates]
    print(f"  người {id_}: importance={e.importance}, cấp dưới trực tiếp={e.subordinates}")
    return e.importance + sum(hand_total(s.id) for s in parts)

print("tổng của 1 =", hand_total(1))

  người 1: importance=5, cấp dưới trực tiếp=[2, 3]
  người 2: importance=3, cấp dưới trực tiếp=[4]
  người 4: importance=-2, cấp dưới trực tiếp=[]
  người 3: importance=3, cấp dưới trực tiếp=[]
tổng của 1 = 9


## 5. Cách 1 — Brute Force / baseline: quét cả danh sách để tìm từng người

### 5.1. Ý tưởng

Chưa có `dict`: mỗi khi cần tìm người có `id = x`, **duyệt lần lượt cả list `employees`** cho tới khi gặp `e.id == x`. Rồi đệ quy: `tổng(x) = e.importance + Σ tổng(s)` với `s` là cấp dưới trực tiếp.

In [9]:
def get_importance_brute(employees, id):
    def find(x):
        for e in employees:                  # quét tuyến tính mỗi lần tra
            if e.id == x:
                return e
        return None

    def total(x):
        e = find(x)
        return e.importance + sum(total(s) for s in e.subordinates)

    return total(id)

for rows, id_, expected in examples:
    got = get_importance_brute(make_employees(rows), id_)
    print(f"id={id_} -> {got}")
    assert got == expected

id=1 -> 11
id=5 -> -3
id=1 -> 12
id=2 -> 7
id=7 -> 10


### 5.2. Độ phức tạp và nút thắt

Mỗi người trong cây con (`m` người) bị tra một lần bằng quét `O(n)`: Time `O(n · m) ≤ O(n²)`, Space `O(độ sâu)` cho đệ quy. Với `n = 2000` thì cỡ `4 · 10^6` bước — chấp nhận được, nhưng lãng phí: ta **tìm đi tìm lại** những người đã biết vị trí. Đo thực tế:

In [10]:
def chain_rows(n):
    """Chuỗi chỉ huy dài n: 1 -> 2 -> ... -> n, importance = 1 (id xếp thứ tự ngược để tránh trùng hợp)."""
    ids = list(range(1, n + 1))
    return [[i, 1, [i + 1] if i < n else []] for i in reversed(ids)]

sys.setrecursionlimit(10000)
for n in (200, 400, 800):
    emps = make_employees(chain_rows(n))
    t0 = time.perf_counter()
    r = get_importance_brute(emps, 1)
    print(f"n={n:4d}: {1000 * (time.perf_counter() - t0):8.2f} ms (kết quả {r})")
sys.setrecursionlimit(1000)

n= 200:     0.70 ms (kết quả 200)
n= 400:     2.28 ms (kết quả 400)
n= 800:     9.21 ms (kết quả 800)


Nút thắt: bước **tra theo `id`** lặp lại tốn `O(n)` mỗi lần. Chỉ cần dựng **một lần** một bảng `id → nhân viên`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — lập chỉ mục.** Dựng `by_id = {e.id: e for e in employees}` một lần (`O(n)`). Từ đó mỗi lần tra một `id` là `O(1)`.

**Bước 2 — duyệt cây con.** Từ `id` được hỏi, đi xuống theo `subordinates`; **cộng `importance` của mỗi người gặp**. Vì quan hệ là cây (mỗi người tối đa một sếp, không chu trình) nên mỗi người trong cây con được gặp **đúng một lần**; không cần tập `visited` (nhưng thêm vào cho an toàn cũng không hại).

**Bước 3 — chọn cách duyệt.** DFS đệ quy (gọn nhất), BFS (hàng đợi), DFS lặp (ngăn xếp). Cả ba đều `O(n)`.

**Bước 4 — tính chung cho nhiều truy vấn.** Nếu cần trả lời **nhiều** `id`, ta tính sẵn `total[v]` cho **mọi** `v` bằng cách xử lý từ lá lên gốc (mỗi người cộng tổng của mình vào sếp). Mỗi truy vấn sau đó là `O(1)`.

**Cạm bẫy cần chủ động tránh:** (1) `id` không phải chỉ số list; (2) `importance` có thể âm — không dùng "lớn nhất"/"bỏ qua số âm"; (3) đệ quy sâu tới 2000 tầng có thể vượt giới hạn mặc định của Python.

## 7. Các cách tiếp theo

Tổng cộng có 5 cách: Cách 1 (quét danh sách, mục 5) và bốn cách dưới đây. Mỗi cách: giải thích từng đoạn → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Bảng băm + DFS đệ quy

Giải thích từng đoạn:

1. `d = {e.id: e for e in employees}`.
2. `dfs(i)` trả về `d[i].importance + sum(dfs(j) for j in d[i].subordinates)`.
3. Trả về `dfs(id)`.

Ngắn gọn, đúng với cấu trúc cây. Chú ý độ sâu đệ quy.

In [11]:
def dfs_demo(employees, id):
    d = {e.id: e for e in employees}
    def dfs(i, depth):
        e = d[i]
        print("  " * depth + f"dfs({i}): importance={e.importance}, cấp dưới={e.subordinates}")
        total = e.importance + sum(dfs(j, depth + 1) for j in e.subordinates)
        print("  " * depth + f"-> tổng({i}) = {total}")
        return total
    return dfs(id, 0)

print(dfs_demo(make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]), 1))

dfs(1): importance=5, cấp dưới=[2, 3]
  dfs(2): importance=3, cấp dưới=[4]
    dfs(4): importance=-2, cấp dưới=[]
    -> tổng(4) = -2
  -> tổng(2) = 1
  dfs(3): importance=3, cấp dưới=[]
  -> tổng(3) = 3
-> tổng(1) = 9
9


In [12]:
class Sol2:
    """Cách 2: bảng băm + DFS đệ quy."""
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        d = {e.id: e for e in employees}

        def dfs(i: int) -> int:
            return d[i].importance + sum(dfs(j) for j in d[i].subordinates)

        return dfs(id)

for rows, id_, expected in examples:
    assert Sol2().getImportance(make_employees(rows), id_) == expected
print("Sol2 OK")

Sol2 OK


**Độ phức tạp:** Time `O(n)` (dựng `dict` `O(n)`, duyệt cây con `O(m) ≤ O(n)`), Space `O(n)` cho `dict` cộng `O(h)` cho ngăn xếp đệ quy (`h` là độ sâu, tối đa `n`).

### 7.2. Cách 3 — Bảng băm + BFS (hàng đợi)

Giải thích từng đoạn:

1. Dựng `d` như cũ; `queue = deque([id])`; `total = 0`.
2. Lặp: `popleft` một `id`, `e = d[id]`, `total += e.importance`, rồi `queue.extend(e.subordinates)`.
3. Hết hàng đợi thì trả `total`.

Không đệ quy, duyệt theo tầng (sếp, rồi cấp dưới trực tiếp, rồi cấp dưới của họ...).

In [13]:
def bfs_demo(employees, id):
    d = {e.id: e for e in employees}
    queue, total = deque([id]), 0
    while queue:
        v = queue.popleft()
        e = d[v]
        total += e.importance
        queue.extend(e.subordinates)
        print(f"  lấy {v}: +{e.importance} -> total={total}, queue={list(queue)}")
    return total

print(bfs_demo(make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]), 1))

  lấy 1: +5 -> total=5, queue=[2, 3]
  lấy 2: +3 -> total=8, queue=[3, 4]
  lấy 3: +3 -> total=11, queue=[4]
  lấy 4: +-2 -> total=9, queue=[]
9


In [14]:
class Sol3:
    """Cách 3: bảng băm + BFS."""
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        d = {e.id: e for e in employees}
        queue = deque([id])
        total = 0
        while queue:
            e = d[queue.popleft()]
            total += e.importance
            queue.extend(e.subordinates)
        return total

for rows, id_, expected in examples:
    assert Sol3().getImportance(make_employees(rows), id_) == expected
print("Sol3 OK")

Sol3 OK


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (hàng đợi tối đa bằng số người ở một tầng, tệ nhất `O(n)`).

### 7.3. Cách 4 — DFS lặp bằng ngăn xếp (có `visited` bảo hiểm)

Giống BFS nhưng lấy từ **cuối** (`pop`) thay vì đầu. Ta thêm tập `seen` để mỗi người chỉ cộng **một lần** — dữ liệu hợp lệ không cần, nhưng nếu dữ liệu bị lỗi (ví dụ một người xuất hiện ở danh sách cấp dưới của hai sếp) thì không cộng trùng và không lặp vô hạn.

1. `stack = [id]`, `seen = set()`, `total = 0`.
2. Lặp tới khi `stack` rỗng: `v = stack.pop()`; nếu `v in seen` thì bỏ qua; ngược lại `seen.add(v)`, `total += d[v].importance`, `stack.extend(d[v].subordinates)`.

In [15]:
def stack_demo(employees, id):
    d = {e.id: e for e in employees}
    stack, seen, total = [id], set(), 0
    while stack:
        v = stack.pop()
        if v in seen:
            continue
        seen.add(v)
        total += d[v].importance
        stack.extend(d[v].subordinates)
        print(f"  lấy {v}: +{d[v].importance} -> total={total}, stack={stack}")
    return total

print(stack_demo(make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]), 1))

  lấy 1: +5 -> total=5, stack=[2, 3]
  lấy 3: +3 -> total=8, stack=[2]
  lấy 2: +3 -> total=11, stack=[4]
  lấy 4: +-2 -> total=9, stack=[]
9


In [16]:
class Sol4:
    """Cách 4: DFS lặp bằng ngăn xếp, có tập seen."""
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        d = {e.id: e for e in employees}
        stack = [id]
        seen = set()
        total = 0
        while stack:
            v = stack.pop()
            if v in seen:
                continue
            seen.add(v)
            total += d[v].importance
            stack.extend(d[v].subordinates)
        return total

for rows, id_, expected in examples:
    assert Sol4().getImportance(make_employees(rows), id_) == expected
print("Sol4 OK")

Sol4 OK


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Ưu điểm lớn: **không đệ quy**, nên chuỗi chỉ huy dài 2000 người vẫn chạy ở giới hạn đệ quy mặc định.

### 7.4. Cách 5 — Tính sẵn tổng cho mọi người từ lá lên gốc (Kahn)

Ý tưởng khác hẳn: thay vì xuống từ `id`, ta tính `total[v]` (tổng cây con) cho **mọi** `v` đi **từ lá lên**. Cần biết mỗi người còn **bao nhiêu cấp dưới chưa tính xong** (`pending`) và **sếp** của họ (`boss`).

1. `total[v] = importance[v]`; `pending[v] = len(subordinates[v])`; `boss[s] = v` cho mỗi cấp dưới `s` của `v`.
2. Hàng đợi ban đầu: mọi **lá** (`pending = 0`) — tổng của lá đã đủ.
3. Lấy `v`; nếu có sếp `p = boss[v]`: `total[p] += total[v]`, `pending[p] -= 1`; nếu `pending[p] == 0` thì `p` đã nhận đủ từ mọi cấp dưới, đưa `p` vào hàng đợi.
4. Cuối cùng `total[id]` là đáp án (và `total[v]` cho mọi `v`).

Mỗi người vào hàng đợi đúng một lần, nên `O(n)`; **truy vấn thứ hai trở đi tốn `O(1)`**.

In [17]:
def kahn_demo(employees):
    total = {e.id: e.importance for e in employees}
    pending = {e.id: len(e.subordinates) for e in employees}
    boss = {s: e.id for e in employees for s in e.subordinates}
    queue = deque(v for v, c in pending.items() if c == 0)
    print("lá ban đầu:", list(queue))
    while queue:
        v = queue.popleft()
        p = boss.get(v)
        if p is not None:
            total[p] += total[v]
            pending[p] -= 1
            print(f"  xong {v} (tổng {total[v]}) -> cộng vào sếp {p}: total[{p}]={total[p]}, còn chờ {pending[p]}")
            if pending[p] == 0:
                queue.append(p)
    return total

print(kahn_demo(make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]])))

lá ban đầu: [3, 4]
  xong 3 (tổng 3) -> cộng vào sếp 1: total[1]=8, còn chờ 1
  xong 4 (tổng -2) -> cộng vào sếp 2: total[2]=1, còn chờ 0
  xong 2 (tổng 1) -> cộng vào sếp 1: total[1]=9, còn chờ 0
{1: 9, 2: 1, 3: 3, 4: -2}


In [18]:
class Sol5:
    """Cách 5: tính tổng cây con cho mọi người từ lá lên (Kahn)."""
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        total = {e.id: e.importance for e in employees}
        pending = {e.id: len(e.subordinates) for e in employees}
        boss = {s: e.id for e in employees for s in e.subordinates}
        queue = deque(v for v, c in pending.items() if c == 0)
        while queue:
            v = queue.popleft()
            p = boss.get(v)
            if p is not None:
                total[p] += total[v]
                pending[p] -= 1
                if pending[p] == 0:
                    queue.append(p)
        return total[id]

for rows, id_, expected in examples:
    assert Sol5().getImportance(make_employees(rows), id_) == expected
print("Sol5 OK")

Sol5 OK


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Dùng khi có **nhiều truy vấn** trên cùng một danh sách (xem biến thể 15.3). Lưu ý: cách này cần đúng giả thiết "mỗi người tối đa một sếp, không chu trình" của đề.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 (DFS đệ quy)

Với `employees = [[1,5,[2,3]],[2,3,[4]],[3,3,[]],[4,-2,[]]]`, `id = 1`:

| lời gọi | importance | cấp dưới | giá trị trả về |
|---|---|---|---|
| `dfs(1)` | 5 | 2, 3 | `5 + dfs(2) + dfs(3)` |
| `dfs(2)` | 3 | 4 | `3 + dfs(4)` |
| `dfs(4)` | -2 | — | `-2` |
| (quay lại) `dfs(2)` | | | `3 + (-2) = 1` |
| `dfs(3)` | 3 | — | `3` |
| (quay lại) `dfs(1)` | | | `5 + 1 + 3 = 9` |

Mã `dfs_demo` ở mục 7.1 in đúng cây gọi này (kèm thụt lề theo độ sâu). Ta chạy thêm trên rừng có hai cây để thấy chỉ cây con của `id` được xét:

In [19]:
rows = [[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []], [10, 7, [11]], [11, 1, []]]
print("tổng của 1:", dfs_demo(make_employees(rows), 1))
print()
print("tổng của 10:", dfs_demo(make_employees(rows), 10))

dfs(1): importance=5, cấp dưới=[2, 3]
  dfs(2): importance=3, cấp dưới=[4]
    dfs(4): importance=-2, cấp dưới=[]
    -> tổng(4) = -2
  -> tổng(2) = 1
  dfs(3): importance=3, cấp dưới=[]
  -> tổng(3) = 3
-> tổng(1) = 9
tổng của 1: 9

dfs(10): importance=7, cấp dưới=[11]
  dfs(11): importance=1, cấp dưới=[]
  -> tổng(11) = 1
-> tổng(10) = 8
tổng của 10: 8


Quan sát: khi hỏi `id = 10`, hai cây của `1` không được thăm — chỉ `10` và `11`.

### 8.2. Trace Cách 3 (BFS) — bảng `queue`

Với ví dụ trên, `id = 1`:

| bước | lấy ra | `total` sau | `queue` sau khi thêm cấp dưới |
|---|---|---|---|
| 1 | 1 | 5 | [2, 3] |
| 2 | 2 | 8 | [3, 4] |
| 3 | 3 | 11 | [4] |
| 4 | 4 | 9 | [] |

(Mã `bfs_demo` in đúng bảng này ở mục 7.2.) Thứ tự thăm là theo tầng: `1`, rồi `2, 3`, rồi `4`.

### 8.3. Trace Cách 4 (ngăn xếp) với dữ liệu lỗi (một người có hai sếp)

Đề đảm bảo mỗi người tối đa một sếp, nhưng ta thử một dữ liệu **không hợp lệ** (`3` nằm trong danh sách cấp dưới của cả `1` và `2`) để thấy vai trò của `seen`:

In [20]:
bad_rows = [[1, 5, [2, 3]], [2, 3, [3]], [3, 4, []]]
print("có seen:", Sol4().getImportance(make_employees(bad_rows), 1), "(người 3 chỉ cộng một lần: 5+3+4 = 12)")
print("không seen (DFS đệ quy):", Sol2().getImportance(make_employees(bad_rows), 1), "(người 3 bị cộng hai lần)")

có seen: 12 (người 3 chỉ cộng một lần: 5+3+4 = 12)
không seen (DFS đệ quy): 16 (người 3 bị cộng hai lần)


Với dữ liệu hợp lệ cả hai cho cùng kết quả; `seen` chỉ là tấm lưới an toàn (và chặn vòng lặp vô hạn nếu có chu trình).

### 8.4. Trace Cách 5 (Kahn): `pending` và `total` theo từng bước

`kahn_demo` ở mục 7.4 in từng bước. Thêm một ví dụ có hai nhánh sâu khác nhau để thấy một người chỉ được đưa vào hàng đợi **sau khi mọi cấp dưới đã cộng xong**:

In [21]:
rows = [[1, 1, [2, 3]], [2, 2, [4, 5]], [3, 4, []], [4, 8, []], [5, 16, []]]
tot = kahn_demo(make_employees(rows))
print("tổng cây con mọi người:", tot)

lá ban đầu: [3, 4, 5]
  xong 3 (tổng 4) -> cộng vào sếp 1: total[1]=5, còn chờ 1
  xong 4 (tổng 8) -> cộng vào sếp 2: total[2]=10, còn chờ 1
  xong 5 (tổng 16) -> cộng vào sếp 2: total[2]=26, còn chờ 0
  xong 2 (tổng 26) -> cộng vào sếp 1: total[1]=31, còn chờ 0
tổng cây con mọi người: {1: 31, 2: 26, 3: 4, 4: 8, 5: 16}


Người `2` chỉ vào hàng đợi sau khi cả `4` và `5` đã cộng vào (`pending[2]` về 0); người `1` chỉ vào sau `2` và `3`.

## 9. Code nộp LeetCode

Dán đoạn dưới vào LeetCode (lớp `Employee` do LeetCode cung cấp; chú thích `'Employee'` dùng chuỗi nên không cần import).

In [22]:
class Solution:
    def getImportance(self, employees: List['Employee'], id: int) -> int:
        d = {e.id: e for e in employees}          # tra nhanh nhân viên theo id

        def dfs(i: int) -> int:
            return d[i].importance + sum(dfs(j) for j in d[i].subordinates)

        return dfs(id)


for rows, id_, expected in examples:
    assert Solution().getImportance(make_employees(rows), id_) == expected
print("Solution chạy đúng 5 ví dụ")

Solution chạy đúng 5 ví dụ


Lưu ý: bản đệ quy này gọn và là cách phổ biến nhất; trên LeetCode nó chạy được với giới hạn đệ quy của môi trường chấm. Nếu bạn không chắc về giới hạn đệ quy (ví dụ ở máy cá nhân với chuỗi chỉ huy dài 2000), hãy dùng bản lặp ở Cách 3/4.

## 10. Vì sao thuật toán đúng

Đặt `T(v)` là **tổng `importance` của `v` và mọi cấp dưới trực tiếp/gián tiếp của `v`** theo đề, `S(v)` là tập các cấp dưới trực tiếp.

**Hệ thức.** `T(v) = importance(v) + Σ_{s ∈ S(v)} T(s)`. Lý do: tập "`v` cùng mọi cấp dưới" là hợp rời của `{v}` và các cây con gốc tại từng `s ∈ S(v)`. **Rời nhau** vì mỗi người có tối đa một sếp trực tiếp: một người thuộc cây con của `s` thì không thể đồng thời thuộc cây con của `s'` khác (nếu không, người đó sẽ có hai đường lên khác nhau tới `v`, mâu thuẫn với "tối đa một sếp" — hoặc `s'` nằm trong cây con của `s`, nhưng khi đó `s'` có sếp `v` và sếp khác, mâu thuẫn). **Đệ quy dừng** vì quan hệ không có chu trình (nếu có chu trình thì một người là cấp dưới gián tiếp của chính mình, vô nghĩa với "sếp").

**Cách 2 (DFS)** tính đúng `T(id)` theo hệ thức này (quy nạp theo chiều cao của cây con). **Cách 3 và 4** duyệt đúng tập "cây con của `id`" (mỗi người đúng một lần do cây con rời nhau) và cộng `importance` từng người, nên `Σ importance` bằng `T(id)`; thứ tự thăm không ảnh hưởng vì phép cộng giao hoán. **Cách 5** duy trì bất biến: *khi `v` bị lấy ra khỏi hàng đợi, `total[v] = T(v)`.* Với lá: `total = importance = T`. Với người có cấp dưới: `v` chỉ được đưa vào hàng đợi khi `pending[v] = 0`, tức mọi cấp dưới `s` đã cộng `total[s] = T(s)` vào `total[v]` đúng một lần, nên `total[v] = importance(v) + Σ T(s) = T(v)`.

**Cách 1** tính cùng hệ thức, chỉ khác cách tìm `v` theo `id` (quét tuyến tính).

**Vì sao `dict` đúng:** `id` duy nhất nên `{e.id: e}` không đè lẫn nhau; các `id` trong `subordinates` hợp lệ nên `d[i]` không lỗi `KeyError`.

## 11. Phân tích Time / Space Complexity

Ký hiệu: `n = len(employees)`, `m` = số người trong cây con của `id` (`m ≤ n`), `h` = độ sâu của cây con.

| Cách | Time | Space |
|---|---|---|
| 1. Quét danh sách mỗi lần tra | `O(n · m)` ≤ `O(n²)` | `O(h)` |
| 2. `dict` + DFS đệ quy | `O(n)` | `O(n + h)` |
| 3. `dict` + BFS | `O(n)` | `O(n)` |
| 4. `dict` + DFS lặp | `O(n)` | `O(n)` |
| 5. Tổng cây con cho mọi người (Kahn) | `O(n)` (và `O(1)` mỗi truy vấn sau) | `O(n)` |

Với `n = 2000`, mọi cách `O(n)` chạy chớp nhoáng. Ta đo trên cây ngẫu nhiên lớn nhất đề cho phép (cây nông nên đệ quy an toàn):

In [23]:
def random_forest(rng, n, id_pool=2000, imp_lo=-100, imp_hi=100, root_prob=0.05):
    """Sinh danh sách [[id, importance, [subordinates]], ...] hợp lệ: id duy nhất, mỗi người tối đa một sếp."""
    ids = rng.sample(range(1, id_pool + 1), n)
    subs = {i: [] for i in ids}
    order = ids[:]                              # thứ tự "ra đời": sếp luôn ra đời trước cấp dưới -> không chu trình
    for idx in range(1, n):
        if rng.random() > root_prob:
            boss = order[rng.randrange(idx)]
            subs[boss].append(order[idx])
    rows = [[i, rng.randint(imp_lo, imp_hi), subs[i][:]] for i in ids]
    rng.shuffle(rows)
    for r in rows:
        rng.shuffle(r[2])
    return rows

rng = random.Random(690)
rows = random_forest(rng, 2000)
emps = make_employees(rows)
for name, cls in [("Cách 2", Sol2), ("Cách 3", Sol3), ("Cách 4", Sol4), ("Cách 5", Sol5), ("Solution", Solution)]:
    t0 = time.perf_counter()
    res = cls().getImportance(emps, rows[0][0])
    print(f"{name:<9}: {1000 * (time.perf_counter() - t0):7.3f} ms -> {res}")

Cách 2   :   0.108 ms -> -94
Cách 3   :   0.114 ms -> -94
Cách 4   :   0.072 ms -> -94
Cách 5   :   1.001 ms -> -94
Solution :   0.067 ms -> -94


## 12. Bộ test

### 12.1. Ví dụ đề và edge case

**Oracle độc lập:** thay vì đi **xuống** từ `id`, ta đi **lên**: với mỗi nhân viên `e`, lần theo chuỗi sếp (`boss`) của `e` tới gốc; nếu gặp `id` thì `e` thuộc cây con của `id` và ta cộng `importance[e]`. Không dùng DFS/BFS xuống, không dùng Kahn. Ta gom các cách vào dict `solvers`.

In [24]:
def oracle(employees, id):
    boss = {}
    for e in employees:
        for s in e.subordinates:
            boss[s] = e.id
    total = 0
    for e in employees:
        v = e.id
        while True:                              # đi lên tới gốc (hoặc tới id)
            if v == id:
                total += e.importance
                break
            if v not in boss:
                break
            v = boss[v]
    return total

solvers = {
    "scan": get_importance_brute,
    "dfs": lambda e, i: Sol2().getImportance(e, i),
    "bfs": lambda e, i: Sol3().getImportance(e, i),
    "stack": lambda e, i: Sol4().getImportance(e, i),
    "kahn": lambda e, i: Sol5().getImportance(e, i),
    "leetcode": lambda e, i: Solution().getImportance(e, i),
}

for rows, id_, expected in examples:
    emps = make_employees(rows)
    assert oracle(emps, id_) == expected
    for name, fn in solvers.items():
        assert fn(emps, id_) == expected, (name, rows, id_)
print("Ví dụ đề: OK cho cả 6 solver và oracle")

Ví dụ đề: OK cho cả 6 solver và oracle


In [25]:
edge_cases = [
    ([[1, 0, []]], 1, 0, "một người, importance 0"),
    ([[1, -100, []]], 1, -100, "importance biên -100"),
    ([[2000, 100, []]], 2000, 100, "id biên 2000"),
    ([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]], 2, 3, "lá của ví dụ 1"),
    ([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]], 3, 3, "lá còn lại"),
    ([[1, 2, [5]], [5, -3, []]], 1, -1, "tổng có thể âm"),
    ([[1, -5, [2]], [2, -5, [3]], [3, -5, []]], 1, -15, "toàn số âm"),
    ([[1, 5, [2]], [2, -5, []]], 1, 0, "triệt tiêu về 0"),
    ([[5, 1, [9, 7]], [9, 2, []], [7, 4, [3]], [3, 8, []]], 5, 15, "id thưa, thứ tự lộn xộn"),
    ([[3, 8, []], [7, 4, [3]], [9, 2, []], [5, 1, [9, 7]]], 7, 12, "đổi thứ tự list"),
    ([[1, 1, [2, 3, 4, 5]], [2, 1, []], [3, 1, []], [4, 1, []], [5, 1, []]], 1, 5, "một sếp nhiều cấp dưới"),
    ([[1, 1, [2]], [2, 2, [3]], [3, 3, [4]], [4, 4, []]], 3, 7, "chuỗi, hỏi giữa chuỗi"),
    ([[1, 5, [2]], [2, 3, []], [10, 100, [11]], [11, 50, []]], 10, 150, "rừng: chỉ cây của id"),
    ([[1, 5, [2]], [2, 3, []], [10, 100, [11]], [11, 50, []]], 1, 8, "rừng: cây còn lại"),
]
for rows, id_, expected, note in edge_cases:
    emps = make_employees(rows)
    assert oracle(emps, id_) == expected, ("oracle", note, oracle(emps, id_))
    for name, fn in solvers.items():
        assert fn(emps, id_) == expected, (name, note)
print(f"{len(edge_cases)} edge case: OK")

14 edge case: OK


### 12.2. Phản ví dụ cho các ý tưởng sai

- **Dùng `id` làm chỉ số list** (`employees[id]` hoặc `employees[id - 1]`): sai khi `id` thưa/thứ tự bị xáo.
- **Chỉ cộng cấp dưới trực tiếp** (một tầng): sai khi có cấp dưới gián tiếp.
- **Không cộng chính người được hỏi**: sai ở mọi ví dụ.

In [26]:
def wrong_index(employees, id):
    e = employees[id - 1]                       # tưởng id là vị trí + 1
    return e.importance + sum(employees[s - 1].importance for s in e.subordinates)

def wrong_one_level(employees, id):
    d = {e.id: e for e in employees}
    return d[id].importance + sum(d[s].importance for s in d[id].subordinates)

def wrong_no_self(employees, id):
    d = {e.id: e for e in employees}
    def dfs(i):
        return sum(d[j].importance + dfs(j) for j in d[i].subordinates)
    return dfs(id)

cases = [
    ([[5, 1, [9, 7]], [9, 2, []], [7, 4, [3]], [3, 8, []]], 5),
    ([[1, 5, [2]], [2, 3, [3]], [3, 4, []]], 1),
    ([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]], 1),
]
for rows, id_ in cases:
    emps = make_employees(rows)
    try:
        wi = wrong_index(emps, id_)
    except IndexError:
        wi = "IndexError"
    print(f"id={id_} | đúng={oracle(emps, id_)} | employees[id-1]={wi} | một tầng={wrong_one_level(emps, id_)} | thiếu chính mình={wrong_no_self(emps, id_)}")
assert wrong_one_level(make_employees(cases[1][0]), 1) != 12
assert wrong_no_self(make_employees(cases[2][0]), 1) != 11

id=5 | đúng=15 | employees[id-1]=IndexError | một tầng=7 | thiếu chính mình=14
id=1 | đúng=12 | employees[id-1]=8 | một tầng=8 | thiếu chính mình=7
id=1 | đúng=11 | employees[id-1]=11 | một tầng=11 | thiếu chính mình=6


### 12.3. Đối chiếu mọi cách bằng test vét cạn trên các cây nhỏ

Duyệt **mọi cây nhãn** nhỏ: với `n = 1..6` người, mọi cách chọn sếp cho từng người (mỗi người `i ≥ 1` có sếp là một người **có chỉ số nhỏ hơn** hoặc **không có sếp**), importance lấy từ `{-2, 0, 3}` ngẫu nhiên theo seed; hỏi **mọi** `id`.

In [27]:
from itertools import product

rng = random.Random(690)
checked = 0
for n in range(1, 7):
    # parent[i] = None hoặc chỉ số < i
    for parents in product(*[[None] + list(range(i)) for i in range(n)]):
        subs = [[] for _ in range(n)]
        for i, p in enumerate(parents):
            if p is not None:
                subs[p].append(i + 1)
        # id = i + 1 (dùng id thưa bằng cách nhân 3 và cộng 1 để tránh trùng hợp với chỉ số)
        mp = lambda i: 3 * i + 1
        rows = [[mp(i), rng.choice([-2, 0, 3]), [mp(s - 1) for s in subs[i]]] for i in range(n)]
        rng.shuffle(rows)
        emps = make_employees(rows)
        for r in rows:
            expected = oracle(emps, r[0])
            for name, fn in solvers.items():
                assert fn(emps, r[0]) == expected, (name, rows, r[0])
            checked += 1
print("đã kiểm", checked, "bộ (danh sách, id): tất cả khớp oracle")

đã kiểm 5039 bộ (danh sách, id): tất cả khớp oracle


### 12.4. Test ngẫu nhiên có seed (đúng ràng buộc)

Dùng `random_forest` (id duy nhất trong `1..2000`, importance `-100..100`, mỗi người tối đa một sếp, không chu trình) với `n` từ `1` đến `60`, hỏi một `id` ngẫu nhiên.

In [28]:
rng = random.Random(6900)
for _ in range(2500):
    n = rng.randint(1, 60)
    rows = random_forest(rng, n, id_pool=rng.choice([60, 200, 2000]) if n <= 60 else 2000)
    assert 1 <= len(rows) <= 2000
    assert len({r[0] for r in rows}) == len(rows)
    assert all(1 <= r[0] <= 2000 and -100 <= r[1] <= 100 for r in rows)
    emps = make_employees(rows)
    qid = rng.choice(rows)[0]
    expected = oracle(emps, qid)
    for name, fn in solvers.items():
        assert fn(emps, qid) == expected, (name, rows, qid)
print("2500 test ngẫu nhiên: OK")

2500 test ngẫu nhiên: OK


### 12.5. Test lớn và các cấu trúc cực đoan (chuỗi dài, ngôi sao)

Ba hình dạng tệ nhất:

1. **Chuỗi dài 2000** (độ sâu 2000): kiểm tra giới hạn đệ quy. Các cách đệ quy cần `sys.setrecursionlimit` lớn hơn 2000; các cách lặp (BFS/ngăn xếp/Kahn) chạy với giới hạn mặc định.
2. **Ngôi sao**: một sếp 1999 cấp dưới trực tiếp (độ rộng lớn).
3. **Cây ngẫu nhiên 2000 người**.

In [29]:
chain = make_employees(chain_rows(2000))
star = make_employees([[1, 3, list(range(2, 2001))]] + [[i, i % 7 - 3, []] for i in range(2, 2001)])
rng = random.Random(69000)
big = make_employees(random_forest(rng, 2000))

old = sys.getrecursionlimit()
sys.setrecursionlimit(10000)                       # cho các cách đệ quy
try:
    assert solvers["dfs"](chain, 1) == 2000 == oracle(chain, 1)
    assert solvers["scan"](make_employees(chain_rows(300)), 1) == 300
    assert solvers["leetcode"](chain, 1) == 2000
finally:
    sys.setrecursionlimit(old)
# các cách lặp không cần nâng giới hạn
for name in ("bfs", "stack", "kahn"):
    assert solvers[name](chain, 1) == 2000
    assert solvers[name](chain, 1000) == 1001
print("chuỗi dài 2000: OK")

expected_star = 3 + sum(i % 7 - 3 for i in range(2, 2001))
for name in ("dfs", "bfs", "stack", "kahn", "leetcode"):
    assert solvers[name](star, 1) == expected_star == oracle(star, 1), name
    assert solvers[name](star, 2) == 2 % 7 - 3
print("ngôi sao 1999 cấp dưới: OK")

sys.setrecursionlimit(10000)
try:
    for qid in random.Random(1).sample([e.id for e in big], 40):
        expected = oracle(big, qid)
        for name in ("dfs", "bfs", "stack", "kahn", "leetcode"):
            assert solvers[name](big, qid) == expected, (name, qid)
finally:
    sys.setrecursionlimit(old)
print("cây ngẫu nhiên 2000 người, 40 truy vấn: OK")

chuỗi dài 2000: OK
ngôi sao 1999 cấp dưới: OK


cây ngẫu nhiên 2000 người, 40 truy vấn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `id` như chỉ số của list

**Code sai:**

In [30]:
class WrongSolution1:
    def getImportance(self, employees, id):
        def dfs(i):
            e = employees[i - 1]                       # sai: giả định employees xếp theo id
            return e.importance + sum(dfs(j) for j in e.subordinates)
        return dfs(id)

emps = make_employees([[5, 1, [9]], [9, 2, []]])
try:
    print("sai  :", WrongSolution1().getImportance(emps, 5))
except IndexError as err:
    print("sai  : IndexError -", err)
print("đúng :", Solution().getImportance(emps, 5))
emps2 = make_employees([[2, 10, []], [1, 7, [2]]])
print("sai  :", WrongSolution1().getImportance(emps2, 1), "(chạy được nhưng ra giá trị sai) | đúng:", Solution().getImportance(emps2, 1))

sai  : IndexError - list index out of range
đúng : 3
sai  : 10 (chạy được nhưng ra giá trị sai) | đúng: 17


**Vì sao sai:** thứ tự trong `employees` **không** liên quan tới `id`. Ở ví dụ đầu `employees[4]` không tồn tại; ở ví dụ sau `employees[0]` là người `id = 2` chứ không phải `1`. **Code đúng:** lập `d = {e.id: e for e in employees}` và tra `d[id]`.

### Lỗi 2: Chỉ cộng cấp dưới trực tiếp (bỏ sót gián tiếp)

In [31]:
class WrongSolution2:
    def getImportance(self, employees, id):
        d = {e.id: e for e in employees}
        return d[id].importance + sum(d[s].importance for s in d[id].subordinates)   # chỉ một tầng

emps = make_employees([[1, 5, [2]], [2, 3, [3]], [3, 4, []]])
print("sai  :", WrongSolution2().getImportance(emps, 1), "| đúng:", Solution().getImportance(emps, 1), "(thiếu người 3)")

sai  : 8 | đúng: 12 (thiếu người 3)


**Vì sao sai:** đề yêu cầu cả cấp dưới **gián tiếp**. Phải đi xuống **đệ quy** (hoặc dùng hàng đợi/ngăn xếp) cho tới khi hết cấp dưới. **Code đúng:** `dfs(i)` gọi lại chính nó cho từng cấp dưới.

### Lỗi 3: Sửa (xoá) dữ liệu đầu vào khi duyệt

In [32]:
class WrongSolution3:
    def getImportance(self, employees, id):
        d = {e.id: e for e in employees}
        total = 0
        queue = [id]
        while queue:
            e = d[queue.pop()]
            total += e.importance
            while e.subordinates:
                queue.append(e.subordinates.pop())       # sai: làm rỗng subordinates của đầu vào
        return total

emps = make_employees([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]])
print("lần 1:", WrongSolution3().getImportance(emps, 1))
print("lần 2 (cùng dữ liệu):", WrongSolution3().getImportance(emps, 1), "(chỉ còn 5!)")
print("dữ liệu bị hỏng:", emps)
print("Solution (không sửa dữ liệu):", Solution().getImportance(make_employees([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]]), 1))

lần 1: 11
lần 2 (cùng dữ liệu): 5 (chỉ còn 5!)
dữ liệu bị hỏng: [Employee(1, 5, []), Employee(2, 3, []), Employee(3, 3, [])]
Solution (không sửa dữ liệu): 11


**Vì sao sai:** `pop()` trên `e.subordinates` **làm rỗng** danh sách gốc, nên truy vấn sau trên cùng dữ liệu cho kết quả sai. **Code đúng:** chỉ **đọc** (`extend(e.subordinates)` hoặc duyệt bằng `for`), không sửa đầu vào.

### Lỗi 4: Cộng nhầm `id` thay vì `importance`

In [33]:
class WrongSolution4:
    def getImportance(self, employees, id):
        d = {e.id: e for e in employees}
        def dfs(i):
            return i + sum(dfs(j) for j in d[i].subordinates)      # sai: cộng id
        return dfs(id)

emps = make_employees([[1, 5, [2, 3]], [2, 3, []], [3, 3, []]])
print("sai  :", WrongSolution4().getImportance(emps, 1), "| đúng:", Solution().getImportance(emps, 1))

sai  : 6 | đúng: 11


**Vì sao sai:** cộng nhầm trường dữ liệu (`id` thay cho `importance`) — dễ xảy ra khi tên biến `id` và thuộc tính `e.id` trùng ý nghĩa. **Code đúng:** `d[i].importance`.

### Lỗi 5: Đệ quy quá sâu trên chuỗi chỉ huy dài

In [34]:
chain = make_employees(chain_rows(600))
old = sys.getrecursionlimit()
try:
    sys.setrecursionlimit(200)                      # mô phỏng giới hạn thấp
    try:
        Solution().getImportance(chain, 1)
        print("không lỗi (?)")
    except RecursionError as err:
        print("Solution đệ quy: RecursionError -", err)
    print("Cách lặp (Sol3), cùng giới hạn thấp:", Sol3().getImportance(chain, 1))
finally:
    sys.setrecursionlimit(old)
print("giới hạn đã khôi phục:", sys.getrecursionlimit())

Solution đệ quy: RecursionError - maximum recursion depth exceeded
Cách lặp (Sol3), cùng giới hạn thấp: 600
giới hạn đã khôi phục: 1000


**Vì sao sai:** mỗi tầng đệ quy chiếm một khung gọi hàm; chuỗi dài vượt giới hạn gây `RecursionError`. **Code đúng:** dùng cách lặp (BFS/ngăn xếp) hoặc nâng `sys.setrecursionlimit` (cẩn thận với tràn bộ nhớ ngăn xếp thật sự).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu dạng **"danh sách đối tượng, mỗi đối tượng trỏ tới đối tượng khác bằng `id`"** → luôn lập **`dict {id: đối tượng}`** trước, đừng quét list.
- Hỏi **"tổng/đếm trong cây con"** → duyệt cây con (DFS/BFS). Hỏi **nhiều lần** → tính sẵn cho mọi nút (từ lá lên, hậu tự/Kahn).
- DFS đệ quy ngắn gọn nhất; BFS/ngăn xếp tránh giới hạn đệ quy và rất dễ viết đúng.
- Giá trị có thể **âm** → đừng khởi tạo tổng/max bằng giả định "không âm"; cũng đừng cắt nhánh sớm vì "tổng đã nhỏ".
- Dữ liệu đầu vào chỉ nên **đọc**, không nên sửa (đặc biệt khi hàm có thể được gọi nhiều lần).
- Dùng `seen` khi **không chắc** dữ liệu là cây (chống cộng trùng và vòng lặp vô hạn).
- Luôn thử: một người, người ở lá, người giữa chuỗi, rừng nhiều cây, `id` thưa, importance âm/0, chuỗi dài 2000, ngôi sao 1999.

## 15. Biến thể

### 15.1. Liệt kê các nhân viên thuộc cây con (id) và đếm cấp dưới

Trả về danh sách `id` (đã sắp xếp) gồm người đó và mọi cấp dưới, và **số cấp dưới** (không tính chính họ).

In [35]:
def subtree_ids(employees, id):
    d = {e.id: e for e in employees}
    out, stack = [], [id]
    while stack:
        v = stack.pop()
        out.append(v)
        stack.extend(d[v].subordinates)
    return sorted(out)

def count_subordinates(employees, id):
    return len(subtree_ids(employees, id)) - 1

emps = make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []], [10, 7, [11]], [11, 1, []]])
print(subtree_ids(emps, 1), count_subordinates(emps, 1), subtree_ids(emps, 10), count_subordinates(emps, 4))

rng = random.Random(151)
for _ in range(500):
    rows = random_forest(rng, rng.randint(1, 40))
    emps = make_employees(rows)
    qid = rng.choice(rows)[0]
    boss = {s: e.id for e in emps for s in e.subordinates}
    expected = []
    for e in emps:
        v = e.id
        while v != qid and v in boss:
            v = boss[v]
        if v == qid:
            expected.append(e.id)
    assert subtree_ids(emps, qid) == sorted(expected)
print("subtree_ids khớp oracle đi-lên")

[1, 2, 3, 4] 3 [10, 11] 0
subtree_ids khớp oracle đi-lên


### 15.2. Độ sâu của tổ chức dưới một nhân viên (số cấp)

Độ sâu = số tầng từ `id` xuống người xa nhất (một người không có cấp dưới có độ sâu `1`). Tương tự LeetCode 559 (Maximum Depth of N-ary Tree) và 1376.

In [36]:
def org_depth(employees, id):
    d = {e.id: e for e in employees}
    level, frontier = 0, [id]
    while frontier:
        level += 1
        frontier = [s for v in frontier for s in d[v].subordinates]
    return level

emps = make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]])
print(org_depth(emps, 1), org_depth(emps, 3), org_depth(emps, 2))

rng = random.Random(152)
for _ in range(500):
    rows = random_forest(rng, rng.randint(1, 40))
    emps = make_employees(rows)
    boss = {s: e.id for e in emps for s in e.subordinates}
    def level_of(v):
        k = 1
        while v in boss:
            v = boss[v]
            k += 1
        return k
    qid = rng.choice(rows)[0]
    # độ sâu dưới qid = max (độ sâu tuyệt đối của nút thuộc cây con) - độ sâu tuyệt đối của qid + 1
    inside = subtree_ids(emps, qid)
    assert org_depth(emps, qid) == max(level_of(v) for v in inside) - level_of(qid) + 1
print("org_depth khớp công thức độ sâu tuyệt đối")

3 1 2


org_depth khớp công thức độ sâu tuyệt đối


### 15.3. Nhiều truy vấn: trả về dict `{id: tổng cây con}` của mọi nhân viên (Cách 5)

Dùng Kahn một lần cho cả danh sách, rồi mỗi truy vấn là tra `dict`.

In [37]:
def all_totals(employees):
    total = {e.id: e.importance for e in employees}
    pending = {e.id: len(e.subordinates) for e in employees}
    boss = {s: e.id for e in employees for s in e.subordinates}
    queue = deque(v for v, c in pending.items() if c == 0)
    while queue:
        v = queue.popleft()
        p = boss.get(v)
        if p is not None:
            total[p] += total[v]
            pending[p] -= 1
            if pending[p] == 0:
                queue.append(p)
    return total

emps = make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]])
print(all_totals(emps))

rng = random.Random(153)
for _ in range(500):
    rows = random_forest(rng, rng.randint(1, 50))
    emps = make_employees(rows)
    totals = all_totals(emps)
    for e in emps:
        assert totals[e.id] == Sol4().getImportance(emps, e.id) == oracle(emps, e.id)
print("all_totals khớp từng truy vấn riêng lẻ")

{1: 9, 2: 1, 3: 3, 4: -2}


all_totals khớp từng truy vấn riêng lẻ


### 15.4. Dữ liệu dạng list thô (đề cũ) và chuỗi chỉ huy lên trên

Nếu đề cho `employees` là list `[[id, importance, [subs]], ...]` (không có lớp `Employee`), ta dựng `dict` từ list thô. Ngoài ra "chuỗi chỉ huy lên trên" (từ `id` lên tới gốc) là phép **ngược** của bài này.

In [38]:
def get_importance_raw(rows, id):
    d = {r[0]: r for r in rows}
    stack, total = [id], 0
    while stack:
        v = stack.pop()
        total += d[v][1]
        stack.extend(d[v][2])
    return total

def chain_of_command(employees, id):
    boss = {s: e.id for e in employees for s in e.subordinates}
    out = [id]
    while out[-1] in boss:
        out.append(boss[out[-1]])
    return out

rows = [[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]]
print(get_importance_raw(rows, 1), get_importance_raw(rows, 2), chain_of_command(make_employees(rows), 4))
assert get_importance_raw(rows, 1) == 9

9 1 [4, 2, 1]


### 15.5. Các bài liên quan trên LeetCode (đọc thêm)

- **559. Maximum Depth of N-ary Tree:** độ sâu cây nhiều con (như 15.2).
- **429. N-ary Tree Level Order Traversal:** BFS theo tầng.
- **1376. Time Needed to Inform All Employees:** cùng khung "sếp–cấp dưới" nhưng cho mảng `manager[]`; tìm đường đi nặng nhất.
- **690 vs 1376:** một bài cho `subordinates` (đi xuống), một bài cho `manager` (đi lên — dựng danh sách con trước).
- **337. House Robber III, 543. Diameter of Binary Tree:** các bài tổng/đếm trong cây con.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét danh sách | tìm từng người bằng duyệt tuyến tính | `O(n · m)` | `O(h)` | chỉ để hiểu đề; dùng làm baseline |
| 2. `dict` + DFS đệ quy | `T(v) = importance + Σ T(s)` | `O(n)` | `O(n + h)` | **cách nộp bài chuẩn**, ngắn gọn |
| 3. `dict` + BFS | hàng đợi, cộng khi lấy ra | `O(n)` | `O(n)` | tránh đệ quy, duyệt theo tầng |
| 4. `dict` + DFS lặp | ngăn xếp + `seen` | `O(n)` | `O(n)` | an toàn khi cây rất sâu / dữ liệu có thể lỗi |
| 5. Kahn từ lá lên | tính tổng cây con cho mọi người | `O(n)` (+`O(1)` mỗi truy vấn) | `O(n)` | **nhiều truy vấn** trên cùng danh sách |

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [39]:
emps = make_employees([[1, 4, [2, 3]], [2, -1, [4]], [3, 6, []], [4, 5, []]])
print(Solution().getImportance(emps, 1))
print(Solution().getImportance(emps, 2))
print(Solution().getImportance(emps, 3))
print(Solution().getImportance(emps, 4))

14
4
6
5


**Bài 2 (tính tay).** Với `employees = [[1,5,[2,3]],[2,3,[4]],[3,3,[]],[4,-2,[]]]`, tính tổng của `id = 1` và của `id = 2`. Gán tuple `(tổng_1, tổng_2)` vào `ans2` (đang là `None`).

In [40]:
ans2 = None      # TODO: tuple (tong_1, tong_2)

**Bài 3 (điền TODO).** Hoàn thành `importance_dfs(employees, id)` bằng DFS đệ quy với `dict`.

In [41]:
def importance_dfs(employees, id):
    d = None       # TODO: dict {e.id: e}

    def dfs(i):
        # TODO: trả về d[i].importance + tổng dfs của các cấp dưới trực tiếp
        return None

    return dfs(id)

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_importance(employees, id)` theo **BFS** (Cách 3) với `deque`.

In [42]:
def my_importance(employees, id):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `top_manager(employees)`: trả về `id` của nhân viên có **tổng cây con lớn nhất** (nếu hoà, chọn `id` nhỏ nhất). (b) `leaf_importance(employees, id)`: tổng `importance` của **chỉ các lá** (người không có cấp dưới) trong cây con của `id`. Ví dụ với `[[1,5,[2,3]],[2,3,[4]],[3,3,[]],[4,-2,[]]]` thì `leaf_importance(·, 1) = 3 + (-2) = 1` (lá là 3 và 4).

In [43]:
def top_manager(employees):
    # TODO
    pass

def leaf_importance(employees, id):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: vẽ cây rồi cộng; nhớ có số âm.
- Bài 2: tổng của `2` gồm `2` và `4`.
- Bài 3: dựng `d` bằng dict comprehension.
- Bài 4: `queue = deque([id])`, mỗi lần `popleft` rồi cộng.
- Bài 5: dùng `all_totals` ở mục 15.3 cho (a); (b) duyệt cây con và kiểm tra `not e.subordinates`.

**Mức 2 (vừa):**
- Bài 1: tổng của `2` là `-1 + 5` (vì `4` là cấp dưới của `2`); tổng của `1` là `4 + (tổng của 2) + 6`.
- Bài 3: `d = {e.id: e for e in employees}`; `return d[i].importance + sum(dfs(j) for j in d[i].subordinates)`.
- Bài 4: `e = d[queue.popleft()]; total += e.importance; queue.extend(e.subordinates)`.
- Bài 5(a): `totals = all_totals(employees)`; `min(totals, key=lambda v: (-totals[v], v))`.

**Mức 3 (gần đáp án):**
- Bài 3: xem Cách 2 ở mục 7.1.
- Bài 4: xem Cách 3 ở mục 7.2.
- Bài 5(b): `stack = [id]; while stack: e = d[stack.pop()]; if not e.subordinates: total += e.importance else: stack.extend(e.subordinates)`.

## 19. Đáp án tham khảo

In [44]:
# Bài 1: cây: 1(4) -> 2(-1) -> 4(5); 1 -> 3(6). Tổng 1 = 4 + (-1 + 5) + 6 = 14; tổng 2 = 4; tổng 3 = 6; tổng 4 = 5
emps = make_employees([[1, 4, [2, 3]], [2, -1, [4]], [3, 6, []], [4, 5, []]])
assert [Solution().getImportance(emps, i) for i in (1, 2, 3, 4)] == [14, 4, 6, 5]
print("Bài 1 OK: 14, 4, 6, 5")

# Bài 2
ans2_ref = (9, 1)
emps = make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]])
assert ans2_ref == (Solution().getImportance(emps, 1), Solution().getImportance(emps, 2))
print("Bài 2 OK:", ans2_ref)

Bài 1 OK: 14, 4, 6, 5
Bài 2 OK: (9, 1)


In [45]:
# Bài 3
def importance_dfs_ref(employees, id):
    d = {e.id: e for e in employees}

    def dfs(i):
        return d[i].importance + sum(dfs(j) for j in d[i].subordinates)

    return dfs(id)

# Bài 4
def my_importance_ref(employees, id):
    d = {e.id: e for e in employees}
    queue = deque([id])
    total = 0
    while queue:
        e = d[queue.popleft()]
        total += e.importance
        queue.extend(e.subordinates)
    return total

rng = random.Random(1903)
for _ in range(1500):
    rows = random_forest(rng, rng.randint(1, 40))
    emps = make_employees(rows)
    qid = rng.choice(rows)[0]
    expected = oracle(emps, qid)
    assert importance_dfs_ref(emps, qid) == expected
    assert my_importance_ref(emps, qid) == expected
    assert [e.subordinates[:] for e in emps] == [r[2] for r in rows], "dữ liệu đầu vào không bị sửa"
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [46]:
# Bài 5
def top_manager_ref(employees):
    totals = all_totals(employees)
    return min(totals, key=lambda v: (-totals[v], v))

def leaf_importance_ref(employees, id):
    d = {e.id: e for e in employees}
    stack, total = [id], 0
    while stack:
        e = d[stack.pop()]
        if not e.subordinates:
            total += e.importance
        else:
            stack.extend(e.subordinates)
    return total

emps = make_employees([[1, 5, [2, 3]], [2, 3, [4]], [3, 3, []], [4, -2, []]])
assert leaf_importance_ref(emps, 1) == 1
assert top_manager_ref(emps) == 1
rng = random.Random(1905)
for _ in range(1000):
    rows = random_forest(rng, rng.randint(1, 40))
    emps = make_employees(rows)
    best = max(oracle(emps, e.id) for e in emps)
    assert oracle(emps, top_manager_ref(emps)) == best
    assert top_manager_ref(emps) == min(e.id for e in emps if oracle(emps, e.id) == best)
    qid = rng.choice(rows)[0]
    inside = set(subtree_ids(emps, qid))
    assert leaf_importance_ref(emps, qid) == sum(e.importance for e in emps if e.id in inside and not e.subordinates)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DFS đệ quy (cách nộp):**

```python
class Solution:
    def getImportance(self, employees, id):
        d = {e.id: e for e in employees}

        def dfs(i):
            return d[i].importance + sum(dfs(j) for j in d[i].subordinates)

        return dfs(id)
```

**Template BFS (không đệ quy):**

```python
from collections import deque

d = {e.id: e for e in employees}
queue, total = deque([id]), 0
while queue:
    e = d[queue.popleft()]
    total += e.importance
    queue.extend(e.subordinates)
return total
```

**Câu thần chú:** *lập `dict` theo `id`; tổng(người) = importance + tổng các cấp dưới; chỉ đọc, không sửa dữ liệu; cây sâu thì dùng hàng đợi/ngăn xếp.*

**Checklist trước khi submit:**

- [ ] Đã dựng `dict {e.id: e}` (không dùng `id` làm chỉ số list)?
- [ ] Đã cộng `importance` của **chính người được hỏi**?
- [ ] Đã tính cả cấp dưới **gián tiếp** (đệ quy / hàng đợi)?
- [ ] Cộng `importance` (không phải `id`); `importance` có thể âm?
- [ ] Không sửa `employees` / `subordinates` của đầu vào?
- [ ] Chuỗi chỉ huy dài 2000: đệ quy có đủ giới hạn, hoặc dùng cách lặp?
- [ ] Đã thử: một người, lá, giữa chuỗi, rừng nhiều cây, `id` thưa, importance âm/0, ngôi sao?